# Nassau Candy Distributor – Exploratory Data Analysis
**Factory Reallocation & Shipping Optimization**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

df = pd.read_csv('../data/Nassau_Candy_Distributor.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# ── Basic Info ──
df.info()
print('\nMissing values:\n', df.isnull().sum())

In [ ]:
df.describe(include='all')

## Feature Engineering

In [ ]:
np.random.seed(42)

ship_mode_days  = {'Same Day': 1, 'First Class': 2, 'Second Class': 4, 'Standard Class': 7}
region_penalty  = {'Interior': 1, 'Atlantic': 0, 'Gulf': 1, 'Pacific': 2}
div_penalty     = {'Chocolate': 0, 'Sugar': 0, 'Other': 1}

df['Lead_Time'] = (
    df['Ship Mode'].map(ship_mode_days) +
    df['Region'].map(region_penalty) +
    df['Division'].map(div_penalty) +
    np.random.randint(0, 3, size=len(df))
)

df['Profit_Margin'] = df['Gross Profit'] / df['Sales']

factory_map = {
    ('Chocolate', 'Atlantic'): 'Factory_East',  ('Chocolate', 'Interior'): 'Factory_Central',
    ('Chocolate', 'Gulf')    : 'Factory_South',  ('Chocolate', 'Pacific') : 'Factory_West',
    ('Sugar',     'Atlantic'): 'Factory_East',  ('Sugar',     'Interior'): 'Factory_Central',
    ('Sugar',     'Gulf')    : 'Factory_South',  ('Sugar',     'Pacific') : 'Factory_West',
    ('Other',     'Atlantic'): 'Factory_East',  ('Other',     'Interior'): 'Factory_Central',
    ('Other',     'Gulf')    : 'Factory_South',  ('Other',     'Pacific') : 'Factory_West',
}
df['Current_Factory'] = df.apply(
    lambda r: factory_map.get((r['Division'], r['Region']), 'Factory_Central'), axis=1
)

print(df[['Lead_Time','Profit_Margin','Current_Factory']].describe())

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df['Lead_Time'], bins=15, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Lead Time Distribution')

sns.histplot(df['Sales'], bins=40, kde=True, ax=axes[1], color='coral')
axes[1].set_title('Sales Distribution')

sns.histplot(df['Profit_Margin'], bins=40, kde=True, ax=axes[2], color='mediumseagreen')
axes[2].set_title('Profit Margin Distribution')
plt.tight_layout()
plt.show()

## Bivariate Analysis

In [ ]:
# Lead Time by Ship Mode
plt.figure(figsize=(10, 5))
order = df.groupby('Ship Mode')['Lead_Time'].mean().sort_values().index
sns.boxplot(data=df, x='Ship Mode', y='Lead_Time', order=order, palette='Set2')
plt.title('Lead Time by Ship Mode')
plt.show()

In [ ]:
# Profit Margin by Region
plt.figure(figsize=(10, 5))
sns.violinplot(data=df, x='Region', y='Profit_Margin', palette='muted', inner='box')
plt.title('Profit Margin by Region')
plt.show()

In [ ]:
# Avg Lead Time heatmap: Division x Region
pivot = df.pivot_table(values='Lead_Time', index='Division', columns='Region', aggfunc='mean')
plt.figure(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd')
plt.title('Average Lead Time: Division × Region')
plt.show()

## Correlation Analysis

In [ ]:
num_cols = ['Sales', 'Units', 'Cost', 'Gross Profit', 'Profit_Margin', 'Lead_Time']
corr = df[num_cols].corr()
plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix')
plt.show()

## Route Clustering

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

agg = df.groupby(['Region', 'Ship Mode', 'Division']).agg(
    Avg_Lead_Time=('Lead_Time', 'mean'),
    Avg_Profit_Margin=('Profit_Margin', 'mean'),
    Volume=('Units', 'sum'),
).reset_index()

scaler = StandardScaler()
X_cl = scaler.fit_transform(agg[['Avg_Lead_Time', 'Avg_Profit_Margin', 'Volume']])

# Elbow method
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cl)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('Elbow Method for Optimal k')
plt.axvline(x=4, color='red', linestyle='--', label='Selected k=4')
plt.legend(); plt.show()

km = KMeans(n_clusters=4, random_state=42, n_init=10)
agg['Cluster'] = km.fit_predict(X_cl)

print('Cluster profiles:')
agg.groupby('Cluster')[['Avg_Lead_Time','Avg_Profit_Margin','Volume']].mean().round(3)

In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    agg['Avg_Lead_Time'], agg['Avg_Profit_Margin'],
    c=agg['Cluster'], s=agg['Volume']/50, cmap='tab10', alpha=0.8
)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('Avg Lead Time'); plt.ylabel('Avg Profit Margin')
plt.title('Route Clusters: Speed vs Profitability')
plt.show()

## Model Training & Evaluation

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

feature_cols = ['Division', 'Region', 'Ship Mode', 'Current_Factory',
                'Sales', 'Units', 'Cost', 'Gross Profit', 'Profit_Margin']

df_enc = pd.get_dummies(df[feature_cols + ['Lead_Time']],
                        columns=['Division', 'Region', 'Ship Mode', 'Current_Factory'])
X = df_enc.drop('Lead_Time', axis=1)
y = df_enc['Lead_Time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Linear Regression' : LinearRegression(),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    results[name] = {
        'RMSE': round(np.sqrt(mean_squared_error(y_test, preds)), 3),
        'MAE' : round(mean_absolute_error(y_test, preds), 3),
        'R2'  : round(r2_score(y_test, preds), 3),
    }

pd.DataFrame(results).T

In [ ]:
# Feature Importance (Random Forest)
rf = models['Random Forest']
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Top 15 Feature Importances (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Key Insights
- Ship Mode is the **strongest predictor** of lead time
- Pacific region consistently has higher lead times
- Standard Class + Pacific + Other Division = highest risk combination
- ~25% of routes can benefit from factory reassignment without margin loss